# Estimación de calidad del café arábica
Guía de sustentación CRISP-DM. Las tablas se leen de resultados ejecutados. Para reproducir el ajuste, ejecute la celda de entrenamiento.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
assert (ROOT/'entrenar.py').exists(), 'Abra el cuaderno desde la carpeta del proyecto.'
from descargar_datos import descargar
if not (ROOT/'data/arabica_data_cleaned.csv').exists(): descargar()


## 1 Entendimiento del negocio
Se predice puntuación de taza antes de conocer las calificaciones sensoriales. La unidad es un lote de café arábica. La aplicación recibe once características disponibles antes de catar.

## 2 Entendimiento de los datos
Se documentan las 43 columnas originales y las once entradas seleccionadas. Las reglas se fijan por significado y plausibilidad; no a partir del mínimo y máximo observado.

In [ ]:
original=pd.read_csv('data/arabica_data_cleaned.csv')
print('Filas y columnas sin índice:',len(original),original.shape[1]-1)
display(pd.read_csv('resultados/diccionario_modelo.csv'))
display(pd.read_csv('resultados/diccionario_completo.csv'))

## 3 Preparación
Se excluye una catación totalmente en cero. Se separa por productor antes de aprender imputaciones, escalas y categorías. Se analizan asociaciones en desarrollo y se compara PCA dentro de los ajustes. No aplica balanceo de clases porque el problema es de regresión.

In [ ]:
display(json.loads(Path('resultados/auditoria.json').read_text()))
display(pd.read_csv('resultados/descripcion_numericas.csv'))
display(pd.read_csv('resultados/relaciones.csv'))
display(Image(filename='figuras/distribucion.png'))
display(Image(filename='figuras/correlaciones.png'))

## 4 Modelamiento y evaluación
Se comparan cinco modelos clásicos y tres ensambles mediante validación anidada. El ganador se fija por RMSE medio sin utilizar la prueba para elegirlo.

In [ ]:
# Esta celda reproduce el análisis y sobrescribe los resultados con la misma semilla.
# Puede omitirla si solo desea presentar los resultados entregados.
import subprocess,sys
subprocess.run([sys.executable,'entrenar.py'],check=True)

In [ ]:
display(pd.read_csv('resultados/comparacion_cv.csv'))
display(json.loads(Path('resultados/anova.json').read_text()))
display(pd.read_csv('resultados/tukey.csv'))
display(pd.read_csv('resultados/comparaciones_corregidas.csv'))
display(pd.read_csv('resultados/evaluacion_prueba.csv'))
display(Image(filename='figuras/observado_predicho.png'))

## 5 Despliegue y predicción de datos nuevos
Se ingresa un CSV externo con las once entradas. Los ejemplos son escenarios hipotéticos sin puntuación real: demuestran inferencia, pero no exactitud futura. El año 2026 representa extrapolación respecto al período histórico.

In [ ]:
from inferencia import cargar,predecir
modelo,metadata=cargar()
nuevos=pd.read_csv('data/entradas_nuevas_ejemplo.csv')
predicciones,avisos=predecir(nuevos,modelo,metadata)
display(predicciones)
print(*avisos,sep='\n')

## Aplicación
Desde una terminal de esta carpeta: `streamlit run app.py`. Para publicar, elegir el repositorio en Streamlit Community Cloud, rama main y archivo app.py. La URL pública y las capturas se incorporan después de completar el acceso y comprobar el despliegue.